In [1]:
import torch
import lightning.pytorch as pl
from pytorch_lightning import Trainer
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import RMSE, MAE, QuantileLoss
from torch.utils.data import DataLoader

import pickle

ROOT="/home/jovyan/Sales_Decline_Forecasting/data/prepared/tft_datasets/"

# Load parameters
# with open(ROOT + 'tft_params.pkl', 'rb') as f:
#     params = pickle.load(f)

# Load datasets and dataloaders
train_dataset = TimeSeriesDataSet.load(ROOT + 'training_dataset.tsd')
val_dataset = TimeSeriesDataSet.load(ROOT + 'validation_dataset.tsd')
test_dataset = TimeSeriesDataSet.load(ROOT + 'test_dataset.tsd')

# Create dataloaders using to_dataloader()
train_dataloader = train_dataset.to_dataloader(batch_size=128, num_workers=4, shuffle=False)
val_dataloader = val_dataset.to_dataloader(batch_size=128, num_workers=4, shuffle=False)
test_dataloader = test_dataset.to_dataloader(batch_size=128, num_workers=4, shuffle=False)


In [2]:
# Verify dataset format
assert isinstance(train_dataset, TimeSeriesDataSet), "Datasets must be TimeSeriesDataSet instances"

In [3]:
params = {
    # Data parameters
    'time_idx': 'time_idx',
    'target': 'purchase_amount',
    'group_ids': ['store'],
    
    # Model architecture
    'hidden_size': 32,
    'lstm_layers': 1,
    'attention_heads': 2,
    'dropout_rate': 0.1,
    
    # Training parameters
    'learning_rate': 0.03,
    'num_epochs': 20,
    
    # Quantile settings
    'quantiles': [0.02, 0.25, 0.5, 0.75, 0.98]
}

In [4]:
# class TFTModel(TemporalFusionTransformer):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)

# Initialize TFT model
tft = TemporalFusionTransformer.from_dataset(
    train_dataset,
    hidden_size=params['hidden_size'],
    lstm_layers=params['lstm_layers'],
    attention_head_size=params['attention_heads'],
    dropout=params['dropout_rate'],
    learning_rate=params['learning_rate'],
    loss=QuantileLoss(),
    optimizer="adam",
    reduce_on_plateau_patience=4
)

/opt/conda/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:198: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/conda/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:198: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [5]:
# Create PyTorch Lightning trainer
trainer = pl.Trainer(
    max_epochs=params['num_epochs'],
    gradient_clip_val=0.1,
    accelerator='auto',
    enable_progress_bar=True,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor="val_loss", patience=5),
        pl.callbacks.ModelCheckpoint(monitor="val_loss")
    ]
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [6]:
assert isinstance(tft, TemporalFusionTransformer), "tft is not TemporalFusionTransformer"

In [7]:
#torch.set_float32_matmul_precision('medium')
#torch.set_grad_enabled(True)

In [8]:
def enable_gradients(model):
    for param in model.parameters():
        param.requires_grad = True

In [9]:
#enable_gradients(tft)

In [ ]:
# Train the model
trainer.fit(
    model=tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

You are using a CUDA device ('NVIDIA A100-SXM4-80GB MIG 1g.10gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 299 K 
3  | prescalers                         | ModuleDict                      | 1.8 K 
4  | static_variable_selection          | VariableSelectionNetwork        | 5.4

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]